In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch
from datasets import load_dataset
from tqdm import tqdm

/home/myid/sn07864/Quantization_benchmark/quant-bench-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_name = "Qwen/Qwen2.5-3B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = torch.float16,
    device_map= "auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Failed to load /home/myid/sn07864/Quantization_benchmark/quant-bench-env/lib/python3.10/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/myid/sn07864/Quantization_benchmark/quant-bench-env/lib/python3.10/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/myid/sn07864/Quantization_benchmark/quant-bench-env/lib/python3.10/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/myid/sn07864/Quantization_benchmark/quant-bench-env/lib/python3.10/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
W0813 21:40:05.190000 2702532 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0813 21:40:05.217000 2702532 torch/utils/_pytree.py:630] 

In [4]:
inputs = tokenizer("The capital of France is", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens = 22)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


The capital of France is Paris. The capital of Germany is Berlin. The capital of Italy is Rome. The capital of Spain is Madrid


In [5]:
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

test_data = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split = 'test')

In [6]:
temp_store = []
for dic in test_data:
    temp_store.append(dic['text'])

raw_text = '\n\n'.join(temp_store)

In [7]:
encodings = tokenizer(raw_text, return_tensors='pt').to(model.device)
print(encodings['input_ids'].shape)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors


torch.Size([1, 299078])


In [8]:
encodings = tokenizer(raw_text, return_tensors='pt').to(model.device)
print(encodings['input_ids'].shape)

torch.Size([1, 299078])


# Now we need to measure Perpexity, VRAM, tokens-sec and disck-size

In [9]:
# encodings are stored in encodings['input_ids']

In [10]:
max_length = 1024 # usually qwen has 32k , which is given by model.config.max_position_embeddings
stride = 512 
seq_len = encodings['input_ids'].size(1)

nll_sum = 0.0 
n_tokens = 0 
prev_end_loc = 0 



In [ ]:
chunk = encodings['input_ids'][:, :1024]
outputs = model(chunk, labels = chunk)
print(outputs.loss)

tensor(1.8197, device='cuda:0', grad_fn=<NllLossBackward0>)


In [12]:
labels = chunk.clone()
labels[:, :512] = -100
outputs2 = model(chunk, labels = labels)
print(outputs2.loss)

tensor(1.5185, device='cuda:0', grad_fn=<NllLossBackward0>)
